In [1]:
# Load environment variables from .env file
import os
import warnings
from dotenv import load_dotenv

# Suppress noisy warnings
warnings.filterwarnings("ignore")

# Load the .env file
load_dotenv()

import os
print("cwd =", os.getcwd())
print(".env exists here?", os.path.exists(".env"))


# Verify keys are loaded (don't print actual keys!)
print("OpenAI key loaded:", "OPENAI_API_KEY" in os.environ)
print("Anthropic key loaded:", "ANTHROPIC_API_KEY" in os.environ)
print("Google key loaded:", "GOOGLE_API_KEY" in os.environ)



cwd = c:\Users\khanh\ai-engineering-fordham
.env exists here? True
OpenAI key loaded: True
Anthropic key loaded: False
Google key loaded: True


In [2]:
from openai import OpenAI

# Create a client (automatically uses OPENAI_API_KEY from environment)
openai_client = OpenAI()

# Make a chat completion request
response = openai_client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "user", "content": "What is Python in exactly one sentence?"}
    ]
)

# Extract the response
print("OpenAI Response:")
print(response.choices[0].message.content)

OpenAI Response:
Python is a high-level, interpreted, general-purpose programming language designed for readability and rapid development, featuring dynamic typing, automatic memory management, a large standard library, and a rich ecosystem of third-party packages.


In [3]:
from google import genai

# Create a client
google_client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

# Generate content (yet another API style!)
response = google_client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="What is Python in exactly one sentence?"
)

# Extract the response
print("Google Response:")
print(response.text)

Google Response:
Python is a high-level, interpreted programming language known for its clear syntax and versatility in fields ranging from web development to artificial intelligence.


In [4]:
import litellm

# Same function works with ANY provider!
def ask_llm(prompt: str, model: str = "gpt-5-mini") -> str:
    """Ask any LLM a question using LiteLLM."""
    response = litellm.completion(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [5]:
# Google (same code, just add "gemini/" prefix)
print("Gemini Flash:", ask_llm("Say 'hello' in French", model="gemini/gemini-2.5-flash"))

Gemini Flash: Bonjour!


In [6]:
response = litellm.completion(
    model="gemini/gemini-2.5-flash",
    messages=[
        # System message: sets the behavior/personality
        {"role": "system", "content": "You are a helpful pirate. Always respond like a pirate."},
        # User message: the actual question
        {"role": "user", "content": "What's the weather like?"}
    ]
)

print(response.choices[0].message.content)

Ahoy there, matey! The winds be howlin' a lively tune, and the clouds are gatherin' like a scurvy crew on the horizon! Looks like there might be a bit o' a squall brewin', good for fillin' our sails if we be headin' out for adventure! Or perhaps a calm sea for a spot o' fishing, eh? Best keep an eye on the sky, a pirate never knows what'll blow in!


In [ ]:
#Manual Retry - vòng lặp nếu gặp lỗi 
import time
import random

def call_with_retries(prompt: str, max_retries: int = 5, base_delay: float = 1.0) -> str:
    """
    Call LLM with exponential backoff retry.
    
    Wait times: 1s -> 2s -> 4s -> 8s -> 16s (plus random jitter)
    """
    for attempt in range(1, max_retries + 1):
        try:
            response = litellm.completion(
                model="gpt-5-mini",
                messages=[{"role": "user", "content": prompt}]
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt == max_retries:
                raise  # Give up after max retries
            
            # Exponential backoff with jitter
            delay = base_delay * (2 ** (attempt - 1)) + random.random()
            print(f"Attempt {attempt} failed: {e}. Retrying in {delay:.1f}s...")
            time.sleep(delay)

# Test it
result = call_with_retries("Say 'hello'")
print(result)

In [5]:
#Retry with tenacity - shorten the code
from tenacity import retry, stop_after_attempt, wait_exponential

@retry(
    stop=stop_after_attempt(5),           # Max 5 attempts
    wait=wait_exponential(multiplier=1, min=1, max=60),  # Exponential backoff
    reraise=True                          # Re-raise the exception if all retries fail
)
def robust_llm_call(prompt: str, model: str = "gpt-5-mini") -> str:
    """Call LLM with automatic retries."""
    response = litellm.completion(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# The decorator handles all retry logic!
result = robust_llm_call("What is 2 + 2?")
print(result)

4


In [ ]:
#openai has their own retries
from openai import OpenAI

# Create client with automatic retries
client = OpenAI(
    max_retries=5,  # Automatically retry up to 5 times
    timeout=30.0    # Timeout after 30 seconds
)

# Now all calls through this client will automatically retry!
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)

## 5. What is Pydantic?
Pydantic is Python's most popular data validation library. It lets you:

Define data structures with types
Automatically validate data
Get helpful error messages
Serialize to/from JSON
It's used everywhere in modern Python: FastAPI, LangChain, Django Ninja, and more

### Basic Pydantic

In [7]:

from pydantic import BaseModel, Field

# Define a data structure
class Person(BaseModel):
    name: str  #khác với class thường là bắt buộc phải nói kiểu dữ liệu mong muốn
    age: int = Field(ge=0, le=150)  # Must be 0-150   
    email: str | None = None        # Optional field
#ge - le: greater or equal to, less or equal to.
#gt - lt: greater than, less than
    

# Create an instance - Pydantic validates automatically!
person = Person(name="Alice", age=30, email="alice@example.com")
print(person)
print(f"Name: {person.name}, Age: {person.age}")

name='Alice' age=30 email='alice@example.com'
Name: Alice, Age: 30


In [8]:
# Pydantic catches invalid data
try:
    invalid_person = Person(name="Bob", age=200)  # Age > 150!
except Exception as e:
    print(f"Validation error: {e}")

Validation error: 1 validation error for Person
age
  Input should be less than or equal to 150 [type=less_than_equal, input_value=200, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal


In [9]:
# Pydantic auto-converts types when possible
person = Person(name="Charlie", age="25")  # String "25" -> int 25
print(f"Age is {person.age}, type: {type(person.age)}")

Age is 25, type: <class 'int'>


5.2 Serialization (From/To Python)


In [10]:
# Convert to JSON
person = Person(name="Diana", age=28)
json_str = person.model_dump_json(indent=2)
print("As JSON:")
print(json_str)

As JSON:
{
  "name": "Diana",
  "age": 28,
  "email": null
}


In [11]:
# Parse from JSON
json_data = '{"name": "Eve", "age": 35, "email": "eve@example.com"}'
person = Person.model_validate_json(json_data)
print(f"Parsed: {person.name}, {person.age}")

Parsed: Eve, 35


## 6. Structured Outputs with Pydantic
The problem: LLMs return free-form text that's hard to parse.

Ask "extract info from this review" and you might get:

"The sentiment is positive and the rating is 4.5"
"Rating: 4.5/5, Sentiment: positive"
"Positive review! 4.5 stars."
Solution: Use response_format to get guaranteed JSON structure.

In [6]:
from pydantic import BaseModel, Field
from typing import Literal

# Define the structure we want
class MovieReview(BaseModel):
    """Structured data extracted from a movie review."""
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="The overall sentiment of the review"
    )
    rating: float = Field(
        description="Numeric rating from 1.0 to 5.0",
        ge=1.0,
        le=5.0
    )
    key_points: list[str] = Field(
        description="Main points mentioned in the review (1-5 items)",
        min_length=1,
        max_length=5
    )
    reviewer_name: str | None = Field(
        default=None,
        description="Name of the reviewer if mentioned"
    )

In [14]:
# A sample review to analyze
review_text = """
This movie was absolutely fantastic! The cinematography was stunning, 
and the acting performances were top-notch. I'd give it 4.5 stars. 
The plot kept me engaged from start to finish. Highly recommend!
- Sarah Johnson
"""

# Use LiteLLM with response_format
response = litellm.completion(
    model="gemini/gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": "Extract structured information from movie reviews."
        },
        {
            "role": "user",
            "content": f"Extract information from this review:\n\n{review_text}"
        }
    ],
    response_format=MovieReview  # Tell the LLM to return this structure!
)

# Parse the JSON response into our Pydantic model
review_data = MovieReview.model_validate_json(response.choices[0].message.content)

# Now we have clean, typed data!
print(f"Sentiment: {review_data.sentiment}")
print(f"Rating: {review_data.rating}")
print(f"Key Points: {review_data.key_points}")
print(f"Reviewer: {review_data.reviewer_name}")

Sentiment: positive
Rating: 4.5
Key Points: ['stunning cinematography', 'top-notch acting performances', 'engaging plot']
Reviewer: Sarah Johnson


### Nested Pydantic 

In [15]:
class Actor(BaseModel):
    """Information about an actor."""
    name: str
    role: str

class MovieInfo(BaseModel):
    """Comprehensive movie information."""
    title: str
    year: int = Field(ge=1900, le=2030)
    genre: list[str]
    director: str
    actors: list[Actor]  # Nested model!
    plot_summary: str = Field(max_length=500)

# Use it
response = litellm.completion(
    model="gemini/gemini-2.5-flash",
    messages=[
        {"role": "user", "content": "Give me information about the movie Inception"}
    ],
    response_format=MovieInfo
)

movie = MovieInfo.model_validate_json(response.choices[0].message.content)
print(f"Title: {movie.title} ({movie.year})")
print(f"Director: {movie.director}")
print(f"Genres: {', '.join(movie.genre)}")
print(f"\nActors:")
for actor in movie.actors:
    print(f"  - {actor.name} as {actor.role}")

Title: Inception (2010)
Director: Christopher Nolan
Genres: Science Fiction, Action, Thriller, Mystery

Actors:
  - Leonardo DiCaprio as Dom Cobb
  - Joseph Gordon-Levitt as Arthur
  - Elliot Page as Ariadne
  - Tom Hardy as Eames
  - Cillian Murphy as Robert Fischer


In [ ]:
# Enable debug mode to see the actual error
litellm.set_verbose = True

In [ ]:
# Enable debug mode to see the actual error
import litellm
litellm.set_verbose = True

# Test a single call to see the error
try:
    test_response = litellm.completion(
        model="gemini/gemini-2.5-flash",
        messages=[{"role": "user", "content": "Test"}]
    )
    print("Success!")
except Exception as e:
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {e}")
    import traceback
    traceback.print_exc()

## 7. Async Programming
Problem: If you need to make 100 LLM calls, doing them one-by-one is slow.

Solution: Make them concurrently with async programming.

### 7.1 What is Async?
Think of it like ordering at a restaurant:

- Synchronous: Order one dish, wait for it, eat it, then order the next
- Asynchronous: Order all dishes at once, they arrive as they're ready

Key Python concepts:

async def - Defines an async function

await - Pause here until the result is ready

asyncio.gather() - Run multiple async tasks concurrently


In [7]:
import time

# A list of prompts to process
prompts = [
    "What is the capital of France?",
    "What is the capital of Germany?",
    "What is the capital of Italy?",
    "What is the capital of Spain?",
    "What is the capital of Portugal?",
    "What is the capital of Greece?",
    "What is the capital of Turkey?",
    "What is the capital of Bulgaria?",
    "What is the capital of Romania?",
    "What is the capital of Hungary?",
    "What is the capital of Poland?",
    "What is the capital of Czech Republic?",
]

# Sequential: one at a time
start = time.time()
sequential_results = []
for prompt in prompts:
    response = litellm.completion(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    sequential_results.append(response.choices[0].message.content)
    time.sleep(5)
sequential_time = time.time() - start

print(f"Sequential: {sequential_time:.2f} seconds")
for prompt, result in zip(prompts, sequential_results):
    print(f"  {prompt} -> {result[:50]}...")

Sequential: 82.41 seconds
  What is the capital of France? -> The capital of France is Paris....
  What is the capital of Germany? -> The capital of Germany is Berlin....
  What is the capital of Italy? -> Rome (Italian: Roma)....
  What is the capital of Spain? -> The capital of Spain is Madrid....
  What is the capital of Portugal? -> The capital of Portugal is Lisbon (Portuguese: Lis...
  What is the capital of Greece? -> The capital of Greece is Athens (Greek: Αθήνα, Ath...
  What is the capital of Turkey? -> The capital of Turkey is Ankara. Many people mista...
  What is the capital of Bulgaria? -> The capital of Bulgaria is Sofia....
  What is the capital of Romania? -> The capital of Romania is Bucharest (Romanian: Buc...
  What is the capital of Hungary? -> The capital of Hungary is Budapest....
  What is the capital of Poland? -> The capital of Poland is Warsaw (Polish: Warszawa)...
  What is the capital of Czech Republic? -> The capital of the Czech Republic is Prague (Czech.

In [8]:
import asyncio

# Async function to call LLM
async def async_ask(prompt: str) -> str:
    """Make an async LLM call."""
    response = await litellm.acompletion(  # Note: acompletion, not completion!
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Async function to process all prompts concurrently
async def process_all(prompts: list[str]) -> list[str]:
    """Process all prompts concurrently."""
    tasks = [async_ask(prompt) for prompt in prompts]
    return await asyncio.gather(*tasks)

# Run concurrently
start = time.time()
concurrent_results = await process_all(prompts)
concurrent_time = time.time() - start

print(f"Concurrent: {concurrent_time:.2f} seconds")
print(f"Speedup: {sequential_time / concurrent_time:.1f}x faster!")

Concurrent: 2.68 seconds
Speedup: 30.8x faster!


In [35]:
#litellm._turn_on_debug()

# Tắt chế độ debug và chỉ hiển thị cảnh báo/lỗi
litellm.set_verbose = False

In [31]:
from google import genai
from google.genai import types
from pathlib import Path

# Create client
google_client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

# Generate an image using Nano Banana (Gemini's native image generation)
response = google_client.models.generate_content(
    model="gemini-2.5-flash-image",  # Nano Banana model
    contents=["A cute robot learning to code, digital art style"],
)

# Find and save the image from the response
output_path = Path("temp/robot_coding.png")
output_path.parent.mkdir(exist_ok=True)

for part in response.parts:
    if part.inline_data is not None:
        image = part.as_image()
        image.save(output_path)
        print(f"Image saved to {output_path}")
        break

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-flash-preview-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-flash-preview-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-flash-preview-image\nPlease retry in 43.414726271s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-ap